## Phase 1: Predicting Drug resistance (y = Drugs, X = Mutations)

In [3]:
#Import packages
import pandas as pd

#Read in data
df = pd.read_csv("geno-pheno.dataset.tsv", sep = "\t")

df.head()

/tmp/ipykernel_7280/2676976629.py:5: DtypeWarning: Columns (0: AZT, 1: DDCFoldMatch, 2: TAFFoldMatch) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("geno-pheno.dataset.tsv", sep = "\t")


,RefID,IsolateID,IsolateName,Species,Type,PtID,Method,3TC,3TCFoldMatch,ABC,...,P291,P292,P293,P294,P295,P296,P297,P298,P299,P300
0,756,9918,CA9918,HIV1,Clinical,1391.0,PhenoSense,200.0,>,4.6,...,-,-,-,-,-,-,-,-,-,-
1,756,3832,CA3832,HIV1,Clinical,1433.0,PhenoSense,200.0,>,8.8,...,-,-,.,.,.,.,.,.,.,.
2,756,10464,CA10464,HIV1,Clinical,634.0,PhenoSense,200.0,>,14.0,...,.,.,.,.,.,.,.,.,.,.
3,756,9928,CA9928,HIV1,Clinical,637.0,PhenoSense,200.0,>,6.7,...,-,-,V,-,-,-,K,-,-,-
4,756,4372,CA4372,HIV1,Clinical,1274.0,PhenoSense,200.0,>,7.1,...,-,-,V,-,-,-,-,-,-,-


In [4]:
##Cleaning the data set
df2 = df.copy()
df2.head()

,RefID,IsolateID,IsolateName,Species,Type,PtID,Method,3TC,3TCFoldMatch,ABC,...,P291,P292,P293,P294,P295,P296,P297,P298,P299,P300
0,756,9918,CA9918,HIV1,Clinical,1391.0,PhenoSense,200.0,>,4.6,...,-,-,-,-,-,-,-,-,-,-
1,756,3832,CA3832,HIV1,Clinical,1433.0,PhenoSense,200.0,>,8.8,...,-,-,.,.,.,.,.,.,.,.
2,756,10464,CA10464,HIV1,Clinical,634.0,PhenoSense,200.0,>,14.0,...,.,.,.,.,.,.,.,.,.,.
3,756,9928,CA9928,HIV1,Clinical,637.0,PhenoSense,200.0,>,6.7,...,-,-,V,-,-,-,K,-,-,-
4,756,4372,CA4372,HIV1,Clinical,1274.0,PhenoSense,200.0,>,7.1,...,-,-,V,-,-,-,-,-,-,-


In [5]:
# Dimensionality of data
df2.shape

(2505, 335)

### Exploratory data analysis

In [6]:
#DRop columns we don't need in the training
df2 = df.drop(columns = ["RefID","Species","Type", "Method", "NNRTIDRMs", 'CompleteMutationListAvailable', 'Author','NonDRMs', 'Author','RefYear', 'MedlineID', 'Title',  'PtID'])
df2.head(n = 300)

#Remove Protease positions

df2 = df2.loc[:, ~df2.columns.str.match(r"^P\d+$")]
df2.head()
df2.shape

(2505, 23)

In [7]:
# Explore the data
#df2.describe()

#df2.info()

df2.columns

df2.shape

(2505, 23)

In [8]:
# Find reverse transcriptase columns
[col for col in df2.columns if "RT" in col.upper()]

['NRTIDRMs']

In [9]:
#Check which columns retained
df2.columns

Index(['IsolateID', 'IsolateName', '3TC', '3TCFoldMatch', 'ABC',
       'ABCFoldMatch', 'AZT', 'AZTFoldMatch', 'D4T', 'D4TFoldMatch', 'DDC',
       'DDCFoldMatch', 'DDI', 'DDIFoldMatch', 'FTC', 'FTCFoldMatch', 'ISL',
       'ISLFoldMatch', 'TAF', 'TAFFoldMatch', 'TDF', 'TDFFoldMatch',
       'NRTIDRMs'],
      dtype='str')

In [10]:
#View data 
df2.head()

,IsolateID,IsolateName,3TC,3TCFoldMatch,ABC,ABCFoldMatch,AZT,AZTFoldMatch,D4T,D4TFoldMatch,...,DDIFoldMatch,FTC,FTCFoldMatch,ISL,ISLFoldMatch,TAF,TAFFoldMatch,TDF,TDFFoldMatch,NRTIDRMs
0,9918,CA9918,200.0,>,4.6,=,0.6,=,1.0,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M184V
1,3832,CA3832,200.0,>,8.8,=,3.2,=,1.9,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,>,14.0,=,307,=,6.4,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,>,6.7,=,6.5,=,1.6,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,>,7.1,=,0.8,=,1.3,=,...,=,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"D67N, K70G, M184V, T215F, K219Q"


In [11]:
#Identifying unique mutations in the dataset

df2.dtypes

IsolateID         int64
IsolateName         str
3TC             float64
3TCFoldMatch        str
ABC             float64
ABCFoldMatch        str
AZT              object
AZTFoldMatch        str
D4T             float64
D4TFoldMatch        str
DDC             float64
DDCFoldMatch        str
DDI             float64
DDIFoldMatch        str
FTC             float64
FTCFoldMatch        str
ISL             float64
ISLFoldMatch        str
TAF             float64
TAFFoldMatch        str
TDF             float64
TDFFoldMatch        str
NRTIDRMs            str
dtype: object

In [12]:
# Number of duplicate IsolateIDs
print("Duplicate IsolateIDs:", df2["IsolateID"].duplicated().sum())


Duplicate IsolateIDs: 0


In [13]:
# Abstract drugs
missing = (df2.isnull().sum().to_frame(name="Missing"))

missing["Percent"] = round(missing["Missing"] / len(df2) * 100, 2)

missing.sort_values("Percent", ascending=False)

,Missing,Percent
ISLFoldMatch,2473,98.72
ISL,2473,98.72
TAFFoldMatch,2409,96.17
TAF,2409,96.17
DDCFoldMatch,2036,81.28
DDC,2003,79.96
FTCFoldMatch,1948,77.76
FTC,1948,77.76
NRTIDRMs,512,20.44
TDF,493,19.68


In [14]:
#Measure completeness of Drug data
drug_cols = ["3TC", "ABC", "AZT", "D4T", "DDC", "DDI", "FTC", "ISL", "TAF", "TDF"]

available = df2[drug_cols].notna().sum().sort_values(ascending=False)

print(available)

AZT    2381
D4T    2377
DDI    2377
3TC    2359
ABC    2231
TDF    2012
FTC     557
DDC     502
TAF      96
ISL      32
dtype: int64


In [15]:
#DRop columns we don't need in the training
df_clean = df2.drop(columns = ["3TCFoldMatch", "ABCFoldMatch", "AZTFoldMatch", "D4TFoldMatch", "DDCFoldMatch", "DDIFoldMatch", "FTCFoldMatch", "ISLFoldMatch", "TAFFoldMatch", "TDFFoldMatch"])
df_clean.head(n = 300)

#Drop DRugs with < 1000 complete cells
df_clean = df_clean.drop(columns = ["DDC", "FTC", "ISL", "TAF"])
df_clean.head(n=30)



,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W"
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y"


In [16]:
#Data types for df_clean
print(df_clean.dtypes)

IsolateID        int64
IsolateName        str
3TC            float64
ABC            float64
AZT             object
D4T            float64
DDI            float64
TDF            float64
NRTIDRMs           str
dtype: object


In [17]:
#Save cleaned data 
df_clean.to_csv("geno-pheno_clean.tsv", sep="\t", index=False)
df_clean.head(n=30)

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W"
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y"


In [18]:
# Remove Isolates that lack NRTIDRMs
#df_clean.dropna(subset = ["NRTIDRMs"], inplace= True)
#df_clean

#Replacing isolates that lack NRTIDRMs with empty strings
df_clean["NRTIDRMs"] = df_clean["NRTIDRMs"].fillna("")
df_clean.tail(n = 30)

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs
2475,743974,DOR-clinical-resistance-7,100.0,2.9,0.5,0.8,1.6,0.8,M184V
2476,692618,ISO3,1.2,0.8,1.7,0.9,0.9,1.2,
2477,692622,ISO7,0.9,1.5,1.5,1.4,1.6,1.8,
2478,692619,ISO4,0.4,0.7,0.3,0.6,0.7,0.3,
2479,743968,DOR-clinical-resistance-1,100.0,2.7,0.1,0.7,1.2,0.5,M184V
2480,743969,DOR-clinical-resistance-2,100.0,3.2,0.1,0.6,1.2,0.6,M184V
2481,692621,ISO6,0.4,0.7,0.8,1.0,0.1,0.8,
2482,743971,DOR-clinical-resistance-4,3.1,0.7,0.2,0.7,1.0,0.3,
2483,692616,ISO1,0.4,1.7,0.3,1.3,1.1,1.5,
2484,743973,DOR-clinical-resistance-6,100.0,2.8,0.1,0.5,1.5,0.4,"K65R, M184V"


In [19]:
#Count Number of Isolates - Number reduced after pruning isolates lacking NRTI-DRMs
df_clean["IsolateID"].nunique()

2505

In [20]:
## Mutation List Generation
df_clean['Mutation_List'] = df_clean['NRTIDRMs'].apply(lambda s: [m.strip() for m in s.split(',') if m.strip()])
df_clean["Mutation_List"].head(n=30)
df_clean["Mutation_List"].tail(n = 30)

2475                                              [M184V]
2476                                                   []
2477                                                   []
2478                                                   []
2479                                              [M184V]
2480                                              [M184V]
2481                                                   []
2482                                                   []
2483                                                   []
2484                                        [K65R, M184V]
2485                                              [M184V]
2486                                               [K65R]
2487                                                   []
2488                                                   []
2489    [M41L, E44D, D67N, T69D, L74V, M184V, L210W, T...
2490                                                   []
2491                           [M41L, L74V, L210W, T215Y]
2492    [M41ML

In [21]:
#Save cleaned data 
df_clean.to_csv("geno-pheno_clean.tsv", sep="\t", index=False)
df_clean.head(n=30)

,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs,Mutation_List
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V,[M184V]
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y","[M41L, L74LV, M184V, L210W, T215Y]"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y","[M41L, E44A, D67N, T69D, M184V, L210W, T215Y]"
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y","[M41L, M184V, T215Y]"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q","[D67N, K70G, M184V, T215F, K219Q]"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q","[D67N, K70R, M184V, K219Q]"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...","[E40F, M41L, D67N, V75M, M184V, L210W, T215Y, ..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W","[M41L, D67N, T69D, K70R, V75M, M184V, T215F, K..."
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q","[D67N, K70R, M184V, K219Q]"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y","[M41L, M184V, L210W, T215Y]"


### Calculating Mutation Frequencies

In [22]:
# Preparing for features (Mutations) Matrix generation 
df_clean[["IsolateName","NRTIDRMs", "Mutation_List"]].tail(30)


,IsolateName,NRTIDRMs,Mutation_List
2475,DOR-clinical-resistance-7,M184V,[M184V]
2476,ISO3,,[]
2477,ISO7,,[]
2478,ISO4,,[]
2479,DOR-clinical-resistance-1,M184V,[M184V]
2480,DOR-clinical-resistance-2,M184V,[M184V]
2481,ISO6,,[]
2482,DOR-clinical-resistance-4,,[]
2483,ISO1,,[]
2484,DOR-clinical-resistance-6,"K65R, M184V","[K65R, M184V]"


### Feature Selection

In [23]:
# Mutation frequency calculation
#Expanding comma-separated mutations into separate rows (Long Format)
df_mut_exploded = df_clean.explode('Mutation_List').dropna(subset=['Mutation_List'])
df_mut_exploded.head()

#Mutation Frequency Analysis
total_isolates = df_clean['IsolateID'].nunique()
global_frequencies = df_mut_exploded['Mutation_List'].value_counts().reset_index()
global_frequencies = global_frequencies[
    (global_frequencies['Mutation_List'] != "") 
]
global_frequencies.columns = ['Mutation_List', 'Absolute_Count']

global_frequencies. head(n = 30)
#print(total_isolates)


,Mutation_List,Absolute_Count
0,M184V,1153
1,M41L,999
2,T215Y,887
3,D67N,811
4,L210W,697
5,K70R,474
6,K219Q,364
7,T215F,259
8,T69D,241
9,E44D,211


In [24]:
#Check dimensionality
df_clean.columns

#Check empty
mut_empty = (df_clean["Mutation_List"].str.len() == 0).sum()
mut_empty


512

total_isolate

In [25]:
#Calculate proportionality of mutations
global_frequencies['Global_Frequency_%'] = (global_frequencies['Absolute_Count'] / (total_isolates - mut_empty))* 100
global_frequencies.head(30)


,Mutation_List,Absolute_Count,Global_Frequency_%
0,M184V,1153,57.852484
1,M41L,999,50.125439
2,T215Y,887,44.505770
3,D67N,811,40.692423
4,L210W,697,34.972403
5,K70R,474,23.783241
6,K219Q,364,18.263924
7,T215F,259,12.995484
8,T69D,241,12.092323
9,E44D,211,10.587055


In [26]:
# Filter based on mutation frequency (< 0.4)
retained_mutations = global_frequencies[global_frequencies["Global_Frequency_%"] >= 1.0]
retained_mutations.head()
retained_mutations.tail()

,Mutation_List,Absolute_Count,Global_Frequency_%
37,A62AV,27,1.354742
38,K219KR,27,1.354742
39,K219KN,26,1.304566
40,T215D,23,1.154039
41,K70G,22,1.103864


In [27]:
retained_mutations.shape

(42, 3)

## Subsetting data 

In [28]:
#Data - df_clean
#Importing data
df_clean = pd.read_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/geno-pheno_clean.tsv",sep ="\t")
df_clean.head(n=30)


,IsolateID,IsolateName,3TC,ABC,AZT,D4T,DDI,TDF,NRTIDRMs,Mutation_List
0,9918,CA9918,200.0,4.6,0.6,1.0,1.6,NaN,M184V,['M184V']
1,3832,CA3832,200.0,8.8,3.2,1.9,2.1,NaN,"M41L, L74LV, M184V, L210W, T215Y","['M41L', 'L74LV', 'M184V', 'L210W', 'T215Y']"
2,10464,CA10464,200.0,14.0,307,6.4,2.7,NaN,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y","['M41L', 'E44A', 'D67N', 'T69D', 'M184V', 'L21..."
3,9928,CA9928,200.0,6.7,6.5,1.6,1.4,NaN,"M41L, M184V, T215Y","['M41L', 'M184V', 'T215Y']"
4,4372,CA4372,200.0,7.1,0.8,1.3,1.9,NaN,"D67N, K70G, M184V, T215F, K219Q","['D67N', 'K70G', 'M184V', 'T215F', 'K219Q']"
5,4391,CA4391,200.0,5.9,1.9,1.3,1.7,NaN,"D67N, K70R, M184V, K219Q","['D67N', 'K70R', 'M184V', 'K219Q']"
6,9912,CA9912,200.0,18.0,100,11.0,3.5,NaN,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...","['E40F', 'M41L', 'D67N', 'V75M', 'M184V', 'L21..."
7,9945,CA9945,200.0,29.0,726,10.0,4.1,NaN,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W","['M41L', 'D67N', 'T69D', 'K70R', 'V75M', 'M184..."
8,9916,CA9916,200.0,7.7,7.9,2.3,1.8,NaN,"D67N, K70R, M184V, K219Q","['D67N', 'K70R', 'M184V', 'K219Q']"
9,9944,CA9944,200.0,8.5,14,1.7,1.7,NaN,"M41L, M184V, L210W, T215Y","['M41L', 'M184V', 'L210W', 'T215Y']"


In [29]:
# Fix AZT fold change values, currently object (str) ---> float
df_clean["AZT"] = (df_clean["AZT"].astype(str).str.replace(",","", regex = False).replace ('nan', pd.NA))
df_clean["AZT"] = pd.to_numeric(df_clean["AZT"], errors = "coerce")

df_clean["AZT"].dtypes

dtype('float64')

In [30]:
# Drug Thresholds 
DRUGS = {
    '3TC' : 3.5,
    'ABC' : 4.5,
    'AZT' : 1.9,
    'D4T' : 1.7,
    'DDI' : 1.3,
    'TDF' : 1.4
}

In [31]:
df_clean.columns

Index(['IsolateID', 'IsolateName', '3TC', 'ABC', 'AZT', 'D4T', 'DDI', 'TDF',
       'NRTIDRMs', 'Mutation_List'],
      dtype='str')

In [32]:
#### 3TC subsetting
print("Lamivudine (3TC) subsetting .....")

df_3TC = (df_clean[['IsolateID','NRTIDRMs','Mutation_List','3TC']].rename(columns = {'3TC' : "FoldChange"}).dropna(subset = ["FoldChange"]).reset_index(drop = True))
df_3TC["resistance"] = (df_3TC["FoldChange"]>= 3.5).astype(int)
print(df_3TC.head())
df_3TC.shape

Lamivudine (3TC) subsetting .....
   IsolateID                                     NRTIDRMs  \
0       9918                                        M184V   
1       3832             M41L, L74LV, M184V, L210W, T215Y   
2      10464  M41L, E44A, D67N, T69D, M184V, L210W, T215Y   
3       9928                           M41L, M184V, T215Y   
4       4372              D67N, K70G, M184V, T215F, K219Q   

                                       Mutation_List  FoldChange  resistance  
0                                          ['M184V']       200.0           1  
1       ['M41L', 'L74LV', 'M184V', 'L210W', 'T215Y']       200.0           1  
2  ['M41L', 'E44A', 'D67N', 'T69D', 'M184V', 'L21...       200.0           1  
3                         ['M41L', 'M184V', 'T215Y']       200.0           1  
4        ['D67N', 'K70G', 'M184V', 'T215F', 'K219Q']       200.0           1  


(2359, 5)

In [33]:
#### ABC subsetting
print("Abacavir (ABC) subsetting .....")

df_ABC = (df_clean[['IsolateID','NRTIDRMs', 'Mutation_List','ABC']].rename(columns = {'ABC' : "ABC_FoldChange"}).dropna(subset = ["ABC_FoldChange"]).reset_index(drop = True))
df_ABC["resistance"] = (df_ABC["ABC_FoldChange"]>= 4.5 ).astype(int)
df_ABC.head(n = 30)
df_ABC.shape


Abacavir (ABC) subsetting .....


(2231, 5)

In [34]:
#### AZT subsetting
print("Zidovudine; AKA Azidothymidine (AZT) subsetting .....")

df_AZT = (df_clean[['IsolateID','NRTIDRMs', 'AZT', "Mutation_List"]].rename(columns = {'AZT' : "AZT_FoldChange"}).dropna(subset = ["AZT_FoldChange"]).reset_index(drop = True))
df_AZT["resistance"] = (df_AZT["AZT_FoldChange"]>= 1.9).astype(int)
print(df_AZT.head(n = 20))
df_AZT.shape


Zidovudine; AKA Azidothymidine (AZT) subsetting .....
    IsolateID                                           NRTIDRMs  \
0        9918                                              M184V   
1        3832                   M41L, L74LV, M184V, L210W, T215Y   
2       10464        M41L, E44A, D67N, T69D, M184V, L210W, T215Y   
3        9928                                 M41L, M184V, T215Y   
4        4372                    D67N, K70G, M184V, T215F, K219Q   
5        4391                           D67N, K70R, M184V, K219Q   
6        9912  E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...   
7        9945  M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W   
8        9916                           D67N, K70R, M184V, K219Q   
9        9944                          M41L, M184V, L210W, T215Y   
10       3871                    M41L, L74V, M184V, L210W, T215Y   
11      10474                                 M41L, M184V, T215Y   
12       9919  A62V, S68G, K70G, V75I, F77L, Y115F, F116Y, Q1.

(2381, 5)

In [35]:
#### D4T subsetting
print("Stavudine; Zerit; (D4T) subsetting .....")

df_D4T = (df_clean[['IsolateID','NRTIDRMs', 'D4T', "Mutation_List"]].rename(columns = {'D4T' : "D4T_FoldChange"}).dropna(subset = ["D4T_FoldChange"]).reset_index(drop = True))
df_D4T["resistance"] = (df_D4T["D4T_FoldChange"]>= 1.7).astype(int)
print(df_D4T.head(n = 20))
df_D4T.shape

Stavudine; Zerit; (D4T) subsetting .....
    IsolateID                                           NRTIDRMs  \
0        9918                                              M184V   
1        3832                   M41L, L74LV, M184V, L210W, T215Y   
2       10464        M41L, E44A, D67N, T69D, M184V, L210W, T215Y   
3        9928                                 M41L, M184V, T215Y   
4        4372                    D67N, K70G, M184V, T215F, K219Q   
5        4391                           D67N, K70R, M184V, K219Q   
6        9912  E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...   
7        9945  M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W   
8        9916                           D67N, K70R, M184V, K219Q   
9        9944                          M41L, M184V, L210W, T215Y   
10       3871                    M41L, L74V, M184V, L210W, T215Y   
11      10474                                 M41L, M184V, T215Y   
12       9919  A62V, S68G, K70G, V75I, F77L, Y115F, F116Y, Q1...   
13     

(2377, 5)

In [36]:
#### DDI  subsetting
print("Didanosine; (DDI) subsetting .....")

df_DDI = (df_clean[['IsolateID','NRTIDRMs', 'DDI', "Mutation_List"]].rename(columns = {'DDI' : "DDI_FoldChange"}).dropna(subset = ["DDI_FoldChange"]).reset_index(drop = True))
df_DDI["resistance"] = (df_DDI["DDI_FoldChange"]>= 1.3).astype(int)
print(df_DDI.head(n = 20))
df_DDI.shape

Didanosine; (DDI) subsetting .....
    IsolateID                                           NRTIDRMs  \
0        9918                                              M184V   
1        3832                   M41L, L74LV, M184V, L210W, T215Y   
2       10464        M41L, E44A, D67N, T69D, M184V, L210W, T215Y   
3        9928                                 M41L, M184V, T215Y   
4        4372                    D67N, K70G, M184V, T215F, K219Q   
5        4391                           D67N, K70R, M184V, K219Q   
6        9912  E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...   
7        9945  M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W   
8        9916                           D67N, K70R, M184V, K219Q   
9        9944                          M41L, M184V, L210W, T215Y   
10       3871                    M41L, L74V, M184V, L210W, T215Y   
11      10474                                 M41L, M184V, T215Y   
12       9919  A62V, S68G, K70G, V75I, F77L, Y115F, F116Y, Q1...   
13       2633

(2377, 5)

In [37]:
#### TDF subsetting
print("Tenofovir Disoproxil; Viread (TDF) subsetting .....")

df_TDF = (df_clean[['IsolateID','NRTIDRMs', 'TDF', "Mutation_List"]].rename(columns = {'TDF' : "TDF_FoldChange"}).dropna(subset = ["TDF_FoldChange"]).reset_index(drop = True))
df_TDF["resistance"] = (df_TDF["TDF_FoldChange"]>= 1.4).astype(int)
print(df_TDF.head(n = 20))
df_TDF.shape

Tenofovir Disoproxil; Viread (TDF) subsetting .....
    IsolateID                                        NRTIDRMs  TDF_FoldChange  \
0       10626                  D67N, T69D, K70R, T215F, K219Q             0.9   
1       15983                 M41L, D67N, M184V, L210W, T215Y             0.9   
2       11732                        M41L, L74V, L210W, T215Y             1.4   
3       11961                 D67N, K70R, M184V, T215V, K219Q             0.9   
4       11193           E40F, M41L, D67N, L210W, T215Y, K219N             4.6   
5       24202                     M41L, T69S_SS, L210W, T215Y            28.0   
6       13897                  M41L, A62V, V75T, M184V, T215Y             1.1   
7        4364                              S68G, F116Y, Q151M             0.9   
8       10967                                A62V, L74V, V75T             0.4   
9       16903                               M41L, D67N, T215Y             2.7   
10      34671                                           K

(2012, 5)

In [ ]:
## List of mutations that meet the threshold
relevant_mutations = list(retained_mutations["Mutation_List"])
relevant_mutations

# # Length of relevant mutation list
# print(len(relevant_mutations))
from pathlib import Path

output_file = Path.home() / "Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day3/relevant_mutations.txt"


#Create csv files for Relevant mutations
with open (output_file, "w") as file:
    file.write(r",".join(relevant_mutations))


In [39]:
#Extract only relevant mutations / Mutations that meet threshold
df_3TC["MutationListFiltered"] = df_3TC["Mutation_List"].apply(lambda muts: [m for m in muts if m in relevant_mutations])
df_3TC.head(n = 30) 

df_ABC["MutationListFiltered"] = df_ABC["Mutation_List"].apply(lambda muts: [m for m in muts if m in relevant_mutations])
df_ABC.head(n = 30) 

df_AZT["MutationListFiltered"] = df_AZT["Mutation_List"].apply(lambda muts: [m for m in muts if m in relevant_mutations])
df_AZT.head(n = 30) 

df_D4T["MutationListFiltered"] = df_D4T["Mutation_List"].apply(lambda muts: [m for m in muts if m in relevant_mutations])
df_D4T.head(n = 30) 

df_DDI["MutationListFiltered"] = df_DDI["Mutation_List"].apply(lambda muts: [m for m in muts if m in relevant_mutations])
df_DDI.head(n = 30) 

df_TDF["MutationListFiltered"] = df_TDF["Mutation_List"].apply(lambda muts: [m for m in muts if m in relevant_mutations])
df_TDF.head(n = 30) 

,IsolateID,NRTIDRMs,TDF_FoldChange,Mutation_List,resistance,MutationListFiltered
0,10626,"D67N, T69D, K70R, T215F, K219Q",0.9,"['D67N', 'T69D', 'K70R', 'T215F', 'K219Q']",0,[]
1,15983,"M41L, D67N, M184V, L210W, T215Y",0.9,"['M41L', 'D67N', 'M184V', 'L210W', 'T215Y']",0,[]
2,11732,"M41L, L74V, L210W, T215Y",1.4,"['M41L', 'L74V', 'L210W', 'T215Y']",1,[]
3,11961,"D67N, K70R, M184V, T215V, K219Q",0.9,"['D67N', 'K70R', 'M184V', 'T215V', 'K219Q']",0,[]
4,11193,"E40F, M41L, D67N, L210W, T215Y, K219N",4.6,"['E40F', 'M41L', 'D67N', 'L210W', 'T215Y', 'K2...",1,[]
5,24202,"M41L, T69S_SS, L210W, T215Y",28.0,"['M41L', 'T69S_SS', 'L210W', 'T215Y']",1,[]
6,13897,"M41L, A62V, V75T, M184V, T215Y",1.1,"['M41L', 'A62V', 'V75T', 'M184V', 'T215Y']",0,[]
7,4364,"S68G, F116Y, Q151M",0.9,"['S68G', 'F116Y', 'Q151M']",0,[]
8,10967,"A62V, L74V, V75T",0.4,"['A62V', 'L74V', 'V75T']",0,[]
9,16903,"M41L, D67N, T215Y",2.7,"['M41L', 'D67N', 'T215Y']",1,[]


In [40]:
## Creating .csv files for each drug
df_3TC.to_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_3TC.csv", index=False)
df_ABC.to_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_ABC.csv", index=False)
df_AZT.to_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_AZT.csv", index=False)
df_D4T.to_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_D4T.csv", index=False)
df_DDI.to_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_DDI.csv", index=False)
df_TDF.to_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_TDF.csv", index=False)

In [41]:
df_3TC.columns

Index(['IsolateID', 'NRTIDRMs', 'Mutation_List', 'FoldChange', 'resistance',
       'MutationListFiltered'],
      dtype='str')

### 3TC Training

In [42]:
#Train model for 3TC prediction
# Data ------> ~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_3TC.csv
import pandas as pd

# import data
df_3TC = pd.read_csv("~/Desktop/Eneza/eneza_project/HIV-drug-resistance-prediction-from-viral-sequences/Day4/drug_dfs/df_3TC.csv") #Row numbers excluded
df_3TC.head(n = 30)

,IsolateID,NRTIDRMs,Mutation_List,FoldChange,resistance,MutationListFiltered
0,9918,M184V,['M184V'],200.0,1,[]
1,3832,"M41L, L74LV, M184V, L210W, T215Y","['M41L', 'L74LV', 'M184V', 'L210W', 'T215Y']",200.0,1,[]
2,10464,"M41L, E44A, D67N, T69D, M184V, L210W, T215Y","['M41L', 'E44A', 'D67N', 'T69D', 'M184V', 'L21...",200.0,1,[]
3,9928,"M41L, M184V, T215Y","['M41L', 'M184V', 'T215Y']",200.0,1,[]
4,4372,"D67N, K70G, M184V, T215F, K219Q","['D67N', 'K70G', 'M184V', 'T215F', 'K219Q']",200.0,1,[]
5,4391,"D67N, K70R, M184V, K219Q","['D67N', 'K70R', 'M184V', 'K219Q']",200.0,1,[]
6,9912,"E40F, M41L, D67N, V75M, M184V, L210W, T215Y, K...","['E40F', 'M41L', 'D67N', 'V75M', 'M184V', 'L21...",200.0,1,[]
7,9945,"M41L, D67N, T69D, K70R, V75M, M184V, T215F, K219W","['M41L', 'D67N', 'T69D', 'K70R', 'V75M', 'M184...",200.0,1,[]
8,9916,"D67N, K70R, M184V, K219Q","['D67N', 'K70R', 'M184V', 'K219Q']",200.0,1,[]
9,9944,"M41L, M184V, L210W, T215Y","['M41L', 'M184V', 'L210W', 'T215Y']",200.0,1,[]


In [43]:
### Create X data frame
X = df_3TC[["IsolateID","MutationListFiltered"]]
X.head(n = 5)

,IsolateID,MutationListFiltered
0,9918,[]
1,3832,[]
2,10464,[]
3,9928,[]
4,4372,[]


In [44]:
### Create y data frame
y = df_3TC["resistance"]
y.head(n = 10)

### Assess whether stratification is needed based on target y == df_3TC["resistance"]
y.value_counts() 

resistance
1    1480
0     879
Name: count, dtype: int64

In [45]:
### Split dataset
from sklearn.model_selection import train_test_split

import pandas as pd


X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42)  # fixes the random number generator to start from the same point


## Inspecting Dimensionality of Split Dataset
#Inspect X_train
X_train.shape

# #Inspect X_test
X_test.shape

# #Inspect y_train
y_train.shape

# #Inspect y_test
y_test.shape

(472,)

In [46]:
## Confirm stratification of y_train and y_test
y_train.value_counts()

y_test.value_counts()

resistance
1    288
0    184
Name: count, dtype: int64

In [47]:
### Generate Mutation matrix (Feature Matrix) for X_train
#Load required libraries
from sklearn.preprocessing import MultiLabelBinarizer

#Mutation matrix:
## X_Train matrix 
mlb_features = MultiLabelBinarizer(classes = relevant_mutations)
X_train_array = mlb_features.fit_transform(X_train)
X_train_array.shape

mlb_features.classes_

### Generate Mutation matrix (Feature Matrix) for X_test
## X_Test matrix
X_test_array = mlb_features.transform(X_test)
X_test_array.shape


## y_test and y_train do not need transformation

/home/noel/miniforge3/envs/ml/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['D', 'F', 'I', 'L', 'M', 'a', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'u'] will be ignored
  warnings.warn(


(2, 42)

In [48]:
# X_train = pd.DataFrame(mlb.fit_transform(train_df["Mutation_List"]),columns=mlb.classes_,index=train_df.index)
# X_test = pd.DataFrame(mlb.transform(test_df["Mutation_List"]),columns=mlb.classes_,index=test_df.index)